# Notebook 03 — Daily Data Pipeline (Clean Slate)

Rebuilds the daily training data from raw parquets with three fixes the prior team missed:
1. **Clip net-negative daily aggregates to 0** — prior team dropped `Quantity < 0` rows but
   returns still appear as negative `total_sales` after product-family aggregation (14,525 affected day-product pairs)
2. **Winsorize per-product at 99th percentile** — 886 products have p99 > 10× median;  
   untreated spikes dominate training loss and cause underprediction on normal weeks
3. **UK bank holiday features** — the primary retail demand driver, absent in prior team's data

Also adds: rolling unit price (4-week), day-of-week and week-of-year cyclical encodings.

**Split boundaries match the prior team exactly** (train ≤ 2011-06-08, test > 2011-06-08),
with a validation set carved from the last 15% of training days (≥ 2011-03-15).

Inputs (in `data/raw/`):
- `Year_2009-2010_post.parquet`
- `Year_2010-2011_post.parquet`

Outputs:
- `data/daily_train.parquet`  — train rows only (before val split), no winsorization applied yet
- `data/daily_val.parquet`
- `data/daily_test.parquet`
- `data/daily_train_winsorized.parquet` — winsorized version for model training
- `data/daily_val_winsorized.parquet`
- `data/daily_test_winsorized.parquet`
- `data/winsor_caps.parquet` — per-product 99th-percentile caps (fit on train only)

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

RAW = Path('../data/raw')
OUT = Path('../data')
OUT.mkdir(exist_ok=True)

# Split boundaries — match prior team exactly
TRAIN_END  = pd.Timestamp('2011-06-08')   # train: date <= TRAIN_END
TEST_START = pd.Timestamp('2011-06-09')   # test:  date >= TEST_START

# Val carved from training tail (last 15% of training days)
# Computed below after building the daily grid; hardcoded here for reference
VAL_START_APPROX = pd.Timestamp('2011-03-15')

# Activity filter: replicate prior team (cumulative train sales >= threshold)
ACTIVITY_THRESHOLD = 100  # gives ~2023 products matching their cluster assignments

## 1. Load & merge raw parquets

In [2]:
raw = pd.concat([
    pd.read_parquet(RAW / 'Year_2009-2010_post.parquet'),
    pd.read_parquet(RAW / 'Year_2010-2011_post.parquet'),
], ignore_index=True)

print(f'Raw rows: {len(raw):,}')
print(f'Columns: {raw.columns.tolist()}')
print(f'Date range: {raw["InvoiceDate"].min()} → {raw["InvoiceDate"].max()}')

Raw rows: 1,067,008
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'product_family_name', 'variant_hint']
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


## 2. Filter: UK only, drop returns (Quantity ≤ 0) and zero-price rows

In [3]:
df = raw[raw['Country'] == 'United Kingdom'].copy()
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

df['date'] = df['InvoiceDate'].dt.normalize()
df['total_sales'] = df['Quantity'] * df['Price']

print(f'After UK + quality filter: {len(df):,} rows')
print(f'Unique product families: {df["product_family_name"].nunique():,}')

After UK + quality filter: 958,502 rows
Unique product families: 2,574


## 3. Compute rolling unit price feature

Daily quantity-weighted average unit price per product, then 28-day rolling mean.
Captures promotional discounting signals missing from the prior team's features.
Computed before daily aggregation to keep unit-price semantics correct.

In [4]:
# Quantity-weighted average unit price per product per day
price_daily = (
    df.groupby(['product_family_name', 'date'])
    .apply(lambda g: np.average(g['Price'], weights=g['Quantity']))
    .reset_index(name='unit_price_daily')
)

print(f'Daily price records: {len(price_daily):,}')
print(f'Unit price range: {price_daily["unit_price_daily"].min():.2f} → {price_daily["unit_price_daily"].max():.2f}')

Daily price records: 330,386
Unit price range: 0.00 → 13541.33


## 4. Aggregate to daily level

In [5]:
daily_sparse = (
    df.groupby(['product_family_name', 'date'], as_index=False)
    .agg(total_sales=('total_sales', 'sum'))
)

# FIX 1: clip net-negative day-product aggregates to 0
# Even though Quantity > 0 was filtered at row level, a product family can have
# positive and negative invoices (returns) on the same day → net negative after groupby.
# The prior team had 14,525 such day-product pairs; we clip them.
neg_before = (daily_sparse['total_sales'] < 0).sum()
daily_sparse['total_sales'] = daily_sparse['total_sales'].clip(lower=0)
print(f'Net-negative day-product rows clipped: {neg_before:,}')
print(f'Daily sparse shape: {daily_sparse.shape}')

Net-negative day-product rows clipped: 0
Daily sparse shape: (330386, 3)


## 5. Build full daily grid (fill missing days with 0)

In [6]:
all_dates    = sorted(daily_sparse['date'].unique())
all_products = daily_sparse['product_family_name'].unique()

grid = pd.MultiIndex.from_product(
    [all_products, all_dates],
    names=['product_family_name', 'date']
).to_frame(index=False)

daily = grid.merge(daily_sparse, on=['product_family_name', 'date'], how='left')
daily['total_sales'] = daily['total_sales'].fillna(0.0)

print(f'Grid shape: {daily.shape}  ({len(all_products):,} products × {len(all_dates):,} days)')
print(f'Non-zero rows: {(daily["total_sales"] > 0).sum():,}')
print(f'Max daily sales: {daily["total_sales"].max():.2f}')

Grid shape: (1554696, 3)  (2,574 products × 604 days)


Non-zero rows: 330,386
Max daily sales: 168469.60


## 6. Join rolling price feature onto full grid

In [7]:
daily = daily.merge(price_daily, on=['product_family_name', 'date'], how='left')

# Forward-fill price within each product (zero-sale days have no observed price)
# then compute 28-day rolling mean
daily = daily.sort_values(['product_family_name', 'date'])
daily['unit_price_daily'] = (
    daily.groupby('product_family_name')['unit_price_daily']
    .transform(lambda s: s.ffill().bfill())
)
daily['price_roll28'] = (
    daily.groupby('product_family_name')['unit_price_daily']
    .transform(lambda s: s.rolling(28, min_periods=1).mean())
)

# Drop raw daily price — rolling mean is the model feature
daily = daily.drop(columns=['unit_price_daily'])

print(f'price_roll28 null count: {daily["price_roll28"].isna().sum()}')
print(daily[['product_family_name', 'date', 'total_sales', 'price_roll28']].head(5))

price_roll28 null count: 0
     product_family_name       date  total_sales  price_roll28
0  *Boombox Ipod Classic 2009-12-01          0.0         16.98
1  *Boombox Ipod Classic 2009-12-02          0.0         16.98
2  *Boombox Ipod Classic 2009-12-03          0.0         16.98
3  *Boombox Ipod Classic 2009-12-04          0.0         16.98
4  *Boombox Ipod Classic 2009-12-05          0.0         16.98


## 7. Activity filter

Keep only products with cumulative sales ≥ 100 in the training window (date ≤ 2011-06-08).
Replicates the prior team's filter exactly; their cluster assignments cover these same 2023 products.

In [8]:
train_sales = (
    daily[daily['date'] <= TRAIN_END]
    .groupby('product_family_name')['total_sales']
    .sum()
)
active_products = train_sales[train_sales >= ACTIVITY_THRESHOLD].index

daily = daily[daily['product_family_name'].isin(active_products)].copy()

print(f'Active products (cumulative train sales ≥ {ACTIVITY_THRESHOLD}): {len(active_products):,}')
print(f'Grid after activity filter: {daily.shape}')

Active products (cumulative train sales ≥ 100): 2,036
Grid after activity filter: (1229744, 4)


## 8. Add UK holiday features

Prior team had zero holiday encoding. UK bank holidays drive the primary retail demand
surge in the days surrounding them.

Features added:
-  — signed distance to nearest UK bank holiday (negative = days since,
  positive = days until); capped at ±30. UK bank holidays have zero transactions in this
  dataset (shops close), so  would always be 0 — omitted.
-  — 1 if ISO week 51 or 52 (Christmas shopping surge)


In [9]:
# pip install workalendar
from workalendar.europe import UnitedKingdom

cal = UnitedKingdom()
min_year = daily["date"].dt.year.min()
max_year = daily["date"].dt.year.max()

holiday_dates = set()
for year in range(min_year, max_year + 2):
    for date, _ in cal.holidays(year):
        holiday_dates.add(pd.Timestamp(date))

unique_dates = pd.Series(sorted(daily["date"].unique()), name="date")

def days_to_nearest_holiday(d):
    diffs = [(d - h).days for h in holiday_dates]
    return min(diffs, key=abs)

# Note: is_holiday is intentionally excluded — UK bank holidays have zero transactions
# in this dataset (shops close), so those dates never appear in the daily grid.
# days_to_holiday (signed distance ±30) captures pre/post-holiday demand effects.
# is_christmas_week is ISO-week-based so it works correctly.
date_holiday_df = pd.DataFrame({"date": unique_dates})
date_holiday_df["days_to_holiday"] = date_holiday_df["date"].apply(days_to_nearest_holiday).clip(-30, 30)
date_holiday_df["is_christmas_week"] = date_holiday_df["date"].dt.isocalendar().week.isin([51, 52]).astype(int).values

daily = daily.merge(date_holiday_df, on="date", how="left")

print(f'Christmas-week days: {daily["is_christmas_week"].sum():,}')
print(f'days_to_holiday range: {daily["days_to_holiday"].min()} → {daily["days_to_holiday"].max()}')


Christmas-week days: 26,468
days_to_holiday range: -30 → 30


## 9. Add calendar features

In [10]:
daily['dayofweek']   = daily['date'].dt.dayofweek          # 0=Mon … 6=Sun
daily['month']       = daily['date'].dt.month
iso = daily['date'].dt.isocalendar()
daily['week_of_year'] = iso.week.astype(int)
daily['year']         = iso.year.astype(int)

# Cyclical encodings — prevent ordinal leakage at year boundaries
daily['dow_sin']  = np.sin(2 * np.pi * daily['dayofweek'] / 7)
daily['dow_cos']  = np.cos(2 * np.pi * daily['dayofweek'] / 7)
daily['week_sin'] = np.sin(2 * np.pi * daily['week_of_year'] / 52)
daily['week_cos'] = np.cos(2 * np.pi * daily['week_of_year'] / 52)
daily['month_sin'] = np.sin(2 * np.pi * daily['month'] / 12)
daily['month_cos'] = np.cos(2 * np.pi * daily['month'] / 12)

print('Columns:', daily.columns.tolist())
daily.head(3)

Columns: ['product_family_name', 'date', 'total_sales', 'price_roll28', 'days_to_holiday', 'is_christmas_week', 'dayofweek', 'month', 'week_of_year', 'year', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos']


,product_family_name,date,total_sales,price_roll28,days_to_holiday,is_christmas_week,dayofweek,month,week_of_year,year,dow_sin,dow_cos,week_sin,week_cos,month_sin,month_cos
0,10 COLOUR SPACEBOY PEN,2009-12-01,0.0,0.746,-24,0,1,12,49,2009,0.781831,0.623490,-0.354605,0.935016,-2.449294e-16,1.0
1,10 COLOUR SPACEBOY PEN,2009-12-02,0.0,0.746,-23,0,2,12,49,2009,0.974928,-0.222521,-0.354605,0.935016,-2.449294e-16,1.0
2,10 COLOUR SPACEBOY PEN,2009-12-03,0.0,0.746,-22,0,3,12,49,2009,0.433884,-0.900969,-0.354605,0.935016,-2.449294e-16,1.0


## 10. Chronological split

- **Train**: date ≤ 2011-06-08 AND date < 2011-03-15  (first ~85% of training days)
- **Val**: 2011-03-15 ≤ date ≤ 2011-06-08            (last ~15% of training days)
- **Test**: date > 2011-06-08

Val is carved from the end of the prior team's training window so we can tune hyperparameters
without touching the held-out test set.

In [11]:
# Compute val start from actual training days (15% holdout)
train_days = sorted(daily[daily['date'] <= TRAIN_END]['date'].unique())
n_train    = len(train_days)
val_start  = train_days[int(n_train * 0.85)]

print(f'Total training days  : {n_train}')
print(f'Val start (15% mark) : {val_start.date()}')
print(f'Train/val boundary   : {train_days[int(n_train * 0.85) - 1].date()} | {val_start.date()}')
print(f'Test start           : {TEST_START.date()}')

train = daily[daily['date'] <  val_start].copy()
val   = daily[(daily['date'] >= val_start) & (daily['date'] <= TRAIN_END)].copy()
test  = daily[daily['date'] >  TRAIN_END].copy()

print(f'\nTrain rows : {len(train):,}  |  products: {train["product_family_name"].nunique():,}')
print(f'Val rows   : {len(val):,}  |  products: {val["product_family_name"].nunique():,}')
print(f'Test rows  : {len(test):,}  |  products: {test["product_family_name"].nunique():,}')

assert train['date'].max() < val['date'].min(), 'Train/val leakage'
assert val['date'].max()   < test['date'].min(), 'Val/test leakage'
print('No temporal leakage.')

Total training days  : 447
Val start (15% mark) : 2011-03-15
Train/val boundary   : 2011-03-14 | 2011-03-15
Test start           : 2011-06-09



Train rows : 771,644  |  products: 2,036
Val rows   : 138,448  |  products: 2,036
Test rows  : 319,652  |  products: 2,036
No temporal leakage.


## 11. Winsorize target

FIX 2: Per-product 99th-percentile cap fit on train only, applied to all splits.

The prior team had no outlier treatment across clusters. Median p99/p50 ratio is 8.9×;
886 products have p99 > 10× their median. Untreated spikes dominate MSE/MAPE training
loss, causing underprediction on normal days and inflating evaluation MAPE.

Caps are saved so models and the agent can recover original scale for output.

In [12]:
caps = (
    train.groupby('product_family_name')['total_sales']
    .quantile(0.99)
    .rename('cap_p99')
    .reset_index()
)
caps.to_parquet(OUT / 'winsor_caps.parquet', index=False)

cap_map = caps.set_index('product_family_name')['cap_p99']

def apply_winsor(df, cap_map):
    df = df.copy()
    caps_aligned = df['product_family_name'].map(cap_map)
    df['total_sales_raw'] = df['total_sales']
    df['total_sales'] = np.minimum(df['total_sales'], caps_aligned.fillna(np.inf))
    return df

train_w = apply_winsor(train, cap_map)
val_w   = apply_winsor(val,   cap_map)
test_w  = apply_winsor(test,  cap_map)

n_clipped_train = (train_w['total_sales'] < train_w['total_sales_raw']).sum()
print(f'Train rows clipped by winsorization: {n_clipped_train:,}')
print(f'Cap p99 distribution: median={cap_map.median():.1f}, max={cap_map.max():.1f}')

Train rows clipped by winsorization: 7,110
Cap p99 distribution: median=46.8, max=13839.4


## 12. Save

In [13]:
# Raw (unwinsorized) splits — used for evaluation (MAPE on true values)
train.to_parquet(OUT / 'daily_train.parquet', index=False)
val.to_parquet(OUT / 'daily_val.parquet', index=False)
test.to_parquet(OUT / 'daily_test.parquet', index=False)

# Winsorized splits — used for model training
train_w.drop(columns=['total_sales_raw']).to_parquet(OUT / 'daily_train_winsorized.parquet', index=False)
val_w.drop(columns=['total_sales_raw']).to_parquet(OUT / 'daily_val_winsorized.parquet', index=False)
test_w.drop(columns=['total_sales_raw']).to_parquet(OUT / 'daily_test_winsorized.parquet', index=False)

print('Saved:')
print(f'  daily_train.parquet          {len(train):,} rows')
print(f'  daily_val.parquet            {len(val):,} rows')
print(f'  daily_test.parquet           {len(test):,} rows')
print(f'  daily_train_winsorized       {len(train_w):,} rows')
print(f'  daily_val_winsorized         {len(val_w):,} rows')
print(f'  daily_test_winsorized        {len(test_w):,} rows')
print(f'  winsor_caps.parquet          {len(caps):,} products')
print(f'\nFinal columns: {train.columns.tolist()}')

Saved:
  daily_train.parquet          771,644 rows
  daily_val.parquet            138,448 rows
  daily_test.parquet           319,652 rows
  daily_train_winsorized       771,644 rows
  daily_val_winsorized         138,448 rows
  daily_test_winsorized        319,652 rows
  winsor_caps.parquet          2,036 products

Final columns: ['product_family_name', 'date', 'total_sales', 'price_roll28', 'days_to_holiday', 'is_christmas_week', 'dayofweek', 'month', 'week_of_year', 'year', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos']


## 13. Sanity checks

In [14]:
# Confirm no negatives remain
assert (train['total_sales'] >= 0).all(), 'Negative values in train'
assert (val['total_sales']   >= 0).all(), 'Negative values in val'
assert (test['total_sales']  >= 0).all(), 'Negative values in test'
print('No negative total_sales values.')

# Confirm temporal ordering
assert train['date'].max() < val['date'].min()
assert val['date'].max()   < test['date'].min()
print('Temporal ordering: train < val < test.')

# Confirm activity-filtered product count matches prior team clustering
import os
cl = pd.read_parquet(OUT / 'clustering' / 'clusters_3models.parquet')
our_products = set(train['product_family_name'].unique())
cl_products  = set(cl.index)
print(f'Products in our train: {len(our_products):,}')
print(f'Products in clusters : {len(cl_products):,}')
print(f'In both              : {len(our_products & cl_products):,}')
print(f'In our train only    : {len(our_products - cl_products):,}  (these will be unrouted in modeling)')

# Summary stats by split
for name, df in [('train', train), ('val', val), ('test', test)]:
    print(f'{name}: {df["date"].min().date()} → {df["date"].max().date()}, '
          f'nonzero rate: {(df["total_sales"] > 0).mean():.3f}')

No negative total_sales values.
Temporal ordering: train < val < test.
Products in our train: 2,036
Products in clusters : 2,023
In both              : 2,023
In our train only    : 13  (these will be unrouted in modeling)
train: 2009-12-01 → 2011-03-14, nonzero rate: 0.260
val: 2011-03-15 → 2011-06-08, nonzero rate: 0.242
test: 2011-06-09 → 2011-12-09, nonzero rate: 0.257
